In [8]:
import cv2
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt



### This is OpenCV learning project like a CamScanner clone, built from scratch.

In this project i will include all pipeline stages.

The first stage is preprocessing.
It will include resizing, grayscale, bluring, and edge detection.


In [9]:
from pathlib import Path
import os
def preprocessing_img(img_path: Path):
    root = os.getcwd()
    img = cv.imread(os.path.join(root,img_path))
    if img is None:
        raise FileNotFoundError(f"Could not load image at {os.path.join(root, img_path)}")
    orig = img.copy()

    height = 500
    ratio = img.shape[0] / height
    img = cv.resize(img, (int(img.shape[1] / ratio), height)) # interpolation is not needed as we are about to blur it anyway.

    gray_img = cv.cvtColor(img, cv.COLOR_BGR2GRAY)


    blur_gray_img = cv.GaussianBlur(gray_img, (5, 5), 0)


    low, up = 70, 200
    edged_blur_gray_img = cv.Canny(blur_gray_img, low, up)

    kernel = np.ones((5, 5), np.uint8)

    edged = cv.morphologyEx(edged_blur_gray_img, cv.MORPH_CLOSE, kernel)

    contours, _ = cv.findContours(edged.copy(), cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv.contourArea, reverse=True)
    img_area = img.shape[0] * img.shape[1]
    doc_contour = None
    for c in contours:
        area = cv.contourArea(c)
        if area > 0.90 * img_area:
            continue
        peri = cv.arcLength(c, True)
        approx = cv.approxPolyDP(c, 0.02 * peri, True)
        if len(approx) == 4:
            doc_contour = approx
            break

    return doc_contour, orig, ratio

In [10]:
# highest sum is the bottom-right
#lowest sum is the top-left
def order_points(pts):
    pts = pts.reshape(4, 2)
    rect = np.zeros((4, 2), dtype="float32")

    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)

    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect




In [11]:
def process(image_path: Path):

    contour, orig, ratio = preprocessing_img(image_path)
    if contour is None:
        print(f"No contour found for {image_path}")
        return
    rect = order_points(contour) * ratio
    (tl, tr, br, bl) = rect

    widthA = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
    widthB = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))

    width = int(max(widthA, widthB))
    heightA = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
    heightB = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))

    height = int(max(heightA, heightB))

    dst = np.array([
        [0, 0],
        [width-1, 0],
        [width-1, height-1],
        [0, height-1]
    ], dtype="float32")

    M = cv.getPerspectiveTransform(rect, dst)
    warped = cv.warpPerspective(orig, M, (width, height))
    cv.imshow("warped", warped)
    cv.waitKey(0)



In [14]:
img_path = Path("IMG_4750.png")

img = cv.imread(img_path)
cv.imshow("Image", img)
cv.waitKey(0)

113

In [15]:
process(img_path)

No contour found for IMG_4750.png
